# First ML Pipeline
## Baselines, Linear & Logistic Regression, and Metrics That Don't Lie

This notebook builds a reproducible machine learning pipeline using
the students dataset. It establishes simple regression and
classification baselines, trains Linear and Logistic Regression
models, evaluates them using appropriate metrics, and compares
each real model against its baseline.

# 1. Feature Engineering

To prepare the student data for machine learning, the categorical `class_section` feature is converted into numerical features using one-hot encoding.

The three numerical input features — `study_hours_per_week`, `sleep_hours_per_night`, and `attendance_pct` — are retained as they are. The `class_section` feature contains three categories: A, B, and C. Using `drop_first=True` creates two encoded columns and treats the omitted category as the reference category.

The resulting feature matrix should therefore contain five input features:
- `study_hours_per_week`
- `sleep_hours_per_night`
- `attendance_pct`
- `class_section_B`
- `class_section_C`

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

In [2]:
rng = np.random.default_rng(seed=21)
n = 600

class_section = rng.choice(["A", "B", "C"], size=n, p=[0.34, 0.33, 0.33])
study_hours = rng.normal(10, 3.5, size=n).clip(0, None).round(1)
sleep_hours = rng.normal(7, 1.2, size=n).clip(3, 10)
attendance_pct = rng.normal(85, 10, size=n).clip(40, 100).round(1)

noise = rng.normal(0, 8, size=n)
section_bonus = pd.Series(class_section).map({"A": 0, "B": 0, "C": 4}).values
exam_score = (50 + 2.6*study_hours + 0.15*attendance_pct + section_bonus + noise).clip(0, 100).round(1)

students = pd.DataFrame({
    "student_id": np.arange(1, n + 1),
    "class_section": class_section,
    "study_hours_per_week": study_hours,
    "sleep_hours_per_night": sleep_hours,
    "attendance_pct": attendance_pct,
    "exam_score": exam_score,
})

## 1.1 Dataset Overview

The generated dataset contains student-level information including class section, study hours, sleep hours, attendance percentage, and exam score.

The `student_id` column serves only as an identifier and will not be used as a predictive feature. The `exam_score` column represents the continuous target for the regression task and is therefore excluded from the input feature matrix.

In [3]:

dataset_overview = f"""
### Finding

The dataset contains **{students.shape[0]:,} observations** and
**{students.shape[1]} columns**.

The regression target is **`exam_score`**, while `student_id` is
treated as an identifier rather than a predictive feature.
"""

display(Markdown(dataset_overview))


### Finding

The dataset contains **600 observations** and
**6 columns**.

The regression target is **`exam_score`**, while `student_id` is
treated as an identifier rather than a predictive feature.


## 1.2 Select the Input Features

The input feature matrix `X` is created using the three numerical predictors and the categorical `class_section` variable.

The target variable `exam_score` is not included in `X` because it is the value the regression model will later learn to predict.

In [4]:
X = students[
    [
        "study_hours_per_week",
        "sleep_hours_per_night",
        "attendance_pct",
        "class_section"
    ]
].copy()

## 1.3 One-Hot Encode `class_section`

The categorical `class_section` feature is converted into numerical indicator variables using one-hot encoding.

The encoding uses `drop_first=True`, so one of the three section categories is treated as the reference category. This produces two encoded columns instead of three while retaining all the information required by the linear and logistic regression models.

In [5]:
X = pd.get_dummies(
    X,
    columns=["class_section"],
    drop_first=True
)

In [6]:
feature_matrix_summary = f"""
### 1.4 Finding

The final feature matrix contains **{X.shape[0]:,} observations** and
**{X.shape[1]} input features** after one-hot encoding `class_section`
with `drop_first=True`.

The resulting features are **{", ".join(X.columns)}**.

The original `class_section` column has been replaced by two encoded
section columns, while the three numerical predictors have been retained.
The omitted section serves as the reference category for the encoded
features.
"""

display(Markdown(feature_matrix_summary))


### 1.4 Finding

The final feature matrix contains **600 observations** and
**5 input features** after one-hot encoding `class_section`
with `drop_first=True`.

The resulting features are **study_hours_per_week, sleep_hours_per_night, attendance_pct, class_section_B, class_section_C**.

The original `class_section` column has been replaced by two encoded
section columns, while the three numerical predictors have been retained.
The omitted section serves as the reference category for the encoded
features.


# 2. Regression Train/Test Split

Before training any regression model, the dataset is divided into a training set and a held-out test set.

The training set is used by the models to learn relationships between the input features and `exam_score`, while the test set remains unseen during training and is used to evaluate how well the models generalize to new observations.

A fixed random seed of `42` is used to make the split reproducible. The same split will be used for the regression baseline and Linear Regression models so that their performance can be compared fairly.

## 2.1 Define the Regression Target

The regression task aims to predict the continuous `exam_score` variable from the engineered input features.

The feature matrix `X` was created in Task 1, while `exam_score` is selected as the regression target `y`.

In [7]:
y = students["exam_score"]

## 2.2 Create the Train/Test Split

The feature matrix and regression target are split into training and testing portions using an 80/20 split.

The split uses `random_state=42` so that the same observations are assigned to the training and test sets each time the notebook is executed. This ensures that every regression model evaluated today is tested on exactly the same unseen observations.

In [8]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 2.3 Verify the Split

The resulting training and test sets are checked to confirm that the intended 80/20 split was applied.

The training set should contain 80% of the observations, while the held-out test set should contain the remaining 20%.

In [9]:
train_test_summary = f"""
### Finding

The training set contains **{X_train.shape[0]:,} observations**
and the test set contains **{X_test.shape[0]:,} observations**.

This corresponds to a **{X_train.shape[0] / len(X) * 100:.0f}% training split**
and a **{X_test.shape[0] / len(X) * 100:.0f}% test split**.

The regression split uses **`random_state=42`**, making the assignment
of observations to the training and test sets reproducible.
"""

display(Markdown(train_test_summary))


### Finding

The training set contains **480 observations**
and the test set contains **120 observations**.

This corresponds to a **80% training split**
and a **20% test split**.

The regression split uses **`random_state=42`**, making the assignment
of observations to the training and test sets reproducible.


## 2.4 Verify Feature Dimensions

The feature dimensions of the training and test sets are checked to ensure that both contain the same five engineered input features.

Only the number of observations differs between the two sets.

In [10]:
split_dimensions = f"""
### Finding

The training feature matrix has a shape of **{X_train.shape}**, while
the test feature matrix has a shape of **{X_test.shape}**.

Both sets contain the same **{X_train.shape[1]} input features**, with
the difference in rows resulting from the 80/20 train/test split.
"""

display(Markdown(split_dimensions))


### Finding

The training feature matrix has a shape of **(480, 5)**, while
the test feature matrix has a shape of **(120, 5)**.

Both sets contain the same **5 input features**, with
the difference in rows resulting from the 80/20 train/test split.


## 2.5 Why This Split Will Be Reused

The same `X_train`, `X_test`, `y_train`, and `y_test` objects will be used for both the regression baseline and Linear Regression.

This is necessary for a meaningful comparison. If the two models were evaluated on different test observations, differences in their metrics could be caused by differences in the test data rather than differences in model performance.

Using the same held-out test set ensures an apples-to-apples comparison between the baseline and the real regression model.

# 3. Regression Baseline

A regression baseline provides a simple reference point against which the real regression model can be evaluated.

The baseline uses `DummyRegressor(strategy="mean")`, which predicts the mean `exam_score` observed in the training set for every observation in the test set.

This model deliberately ignores all input features. Its purpose is to establish the minimum performance that a real regression model should improve upon.

## 3.1 Train the Regression Baseline

The `DummyRegressor` is fitted using the regression training data from Task 2.

With the `mean` strategy, the model learns the average `exam_score` from `y_train` and uses this single value as the prediction for every test observation.

The test set remains completely unseen during fitting.

In [11]:

regression_baseline = DummyRegressor(strategy="mean")

regression_baseline.fit(X_train, y_train)

baseline_predictions = regression_baseline.predict(X_test)

## 3.2 Inspect the Baseline Prediction

Because the baseline uses the mean strategy, every test observation receives the same predicted exam score.

The prediction value should equal the mean exam score calculated from the training set.

In [12]:
baseline_prediction_summary = f"""
### Finding

The regression baseline predicts **{regression_baseline.constant_[0][0]:.2f}**
for every observation in the test set.

This value represents the mean `exam_score` learned from the training data,
so the baseline does not use any of the engineered input features to make
individual predictions.
"""

display(Markdown(baseline_prediction_summary))


### Finding

The regression baseline predicts **88.40**
for every observation in the test set.

This value represents the mean `exam_score` learned from the training data,
so the baseline does not use any of the engineered input features to make
individual predictions.


## 3.3 Evaluate the Regression Baseline

The baseline is evaluated on the held-out test set using Root Mean Squared Error (RMSE) and R².

RMSE measures the typical size of the prediction error in the same units as `exam_score`, with lower values indicating better performance.

R² measures the proportion of variation in the target explained by the model relative to the mean-prediction reference. Since this baseline predicts the mean, its R² provides the reference level against which the Linear Regression model will be compared.

In [13]:
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

## 3.4 Baseline Performance

The baseline RMSE and R² are recorded below as the reference performance for the regression task.

These values will be compared directly with the Linear Regression results in Task 4. Both models will be evaluated using the same test observations.

In [14]:
regression_baseline_finding = f"""
### Finding

The regression baseline achieved an **RMSE of {baseline_rmse:.6f}**
and an **R² of {baseline_r2:.6f}** on the held-out test set.

The baseline predicts the training-set mean for every test observation,
so these metrics establish the minimum reference performance that the
Linear Regression model must beat.
"""

display(Markdown(regression_baseline_finding))


### Finding

The regression baseline achieved an **RMSE of 10.297789**
and an **R² of -0.009560** on the held-out test set.

The baseline predicts the training-set mean for every test observation,
so these metrics establish the minimum reference performance that the
Linear Regression model must beat.


# 4. Linear Regression Model

A Linear Regression model is trained using the same engineered features and regression target used by the baseline.

Unlike the baseline, which predicts the same mean `exam_score` for every student, Linear Regression learns relationships between the input features and the target from the training data.

The model is evaluated on the same held-out test set used for the regression baseline so that its performance can be compared fairly.

## 4.1 Train the Linear Regression Model

The Linear Regression model is fitted using `X_train` and `y_train`.

The model learns coefficients for the five engineered features that minimize the squared prediction errors on the training data. After fitting, predictions are generated for the unseen test set.

In [15]:

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

## 4.2 Evaluate the Linear Regression Model

The Linear Regression model is evaluated using the same two metrics used for the regression baseline: RMSE and R².

RMSE measures the typical size of the prediction error in the units of `exam_score`, where lower values indicate better predictions.

R² measures the proportion of variation in `exam_score` explained by the model relative to the mean-prediction reference, where higher values indicate better performance.

In [16]:
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

## 4.3 Comparison with the Regression Baseline

The Linear Regression metrics are compared directly with the baseline metrics calculated in Task 3.

The RMSE improvement is calculated as the reduction in prediction error from the baseline to Linear Regression. The R² improvement is calculated as the increase in explained variation.

In [17]:
rmse_improvement = baseline_rmse - linear_rmse
r2_improvement = linear_r2 - baseline_r2

In [18]:
linear_regression_finding = f"""
### Finding

The Linear Regression model achieved an **RMSE of {linear_rmse:.6f}** and an
**R² of {linear_r2:.6f}**, compared with the baseline RMSE of
**{baseline_rmse:.6f}** and baseline R² of **{baseline_r2:.6f}**.

**Compared with the baseline, Linear Regression reduced RMSE by
{rmse_improvement:.6f} and increased R² by {r2_improvement:.6f}.**
"""

display(Markdown(linear_regression_finding))


### Finding

The Linear Regression model achieved an **RMSE of 7.058469** and an
**R² of 0.525687**, compared with the baseline RMSE of
**10.297789** and baseline R² of **-0.009560**.

**Compared with the baseline, Linear Regression reduced RMSE by
3.239320 and increased R² by 0.535247.**


## 5. Display Coefficients with Feature Names

The Linear Regression coefficients show how much the predicted `exam_score` changes for a one-unit increase in each numerical feature, while holding the other features constant.

For the one-hot encoded section features, each coefficient represents the predicted difference from the omitted reference section, which is Section A.

In [19]:
coefficient_table = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": linear_model.coef_
})

display(coefficient_table)

,feature,coefficient
0,study_hours_per_week,2.032968
1,sleep_hours_per_night,-0.009430
2,attendance_pct,0.088653
3,class_section_B,0.124746
4,class_section_C,3.130411


In [20]:
largest_effect_index = np.argmax(np.abs(linear_model.coef_))

largest_effect_feature = X_train.columns[largest_effect_index]
largest_effect_coefficient = linear_model.coef_[largest_effect_index]

coefficient_finding = f"""
### 5.1 Finding

The feature with the largest absolute coefficient is
**`{largest_effect_feature}`**, with a coefficient of
**{largest_effect_coefficient:.6f}**.

This means that, holding the other features constant, a one-unit increase in
`{largest_effect_feature}` is associated with an estimated change of
**{largest_effect_coefficient:.6f}** points in the predicted exam score.

The largest effect is therefore associated with **study hours per week** among
the numerical predictors, while `class_section_C` also has a noticeable
positive effect relative to the reference Section A.
"""

display(Markdown(coefficient_finding))


### 5.1 Finding

The feature with the largest absolute coefficient is
**`class_section_C`**, with a coefficient of
**3.130411**.

This means that, holding the other features constant, a one-unit increase in
`class_section_C` is associated with an estimated change of
**3.130411** points in the predicted exam score.

The largest effect is therefore associated with **study hours per week** among
the numerical predictors, while `class_section_C` also has a noticeable
positive effect relative to the reference Section A.


In [21]:
study_hours_correlation = students["study_hours_per_week"].corr(
    students["exam_score"]
)

attendance_correlation = students["attendance_pct"].corr(
    students["exam_score"]
)

sleep_correlation = students["sleep_hours_per_night"].corr(
    students["exam_score"]
)

comparison_finding = f"""
### 5.2 Finding

The coefficient analysis is consistent with Monday's correlation findings.
`study_hours_per_week` had the strongest Pearson correlation with
`exam_score`, with **r = {study_hours_correlation:.6f}**.

The Linear Regression model also assigns the strongest coefficient among the
continuous predictors to `study_hours_per_week`, with a coefficient of
**{linear_model.coef_[0]:.6f}**.

By comparison, `attendance_pct` had a weaker correlation with exam score
(**r = {attendance_correlation:.6f}**) and a smaller regression coefficient
of **{linear_model.coef_[2]:.6f}**. `sleep_hours_per_night` was essentially
uncorrelated with exam score (**r = {sleep_correlation:.6f}**) and its
regression coefficient is also close to zero.

Therefore, the main conclusion from Monday's correlation analysis is
supported by the Linear Regression model: **study hours are the strongest
continuous predictor of exam score in this dataset.**
"""

display(Markdown(comparison_finding))


### 5.2 Finding

The coefficient analysis is consistent with Monday's correlation findings.
`study_hours_per_week` had the strongest Pearson correlation with
`exam_score`, with **r = 0.689476**.

The Linear Regression model also assigns the strongest coefficient among the
continuous predictors to `study_hours_per_week`, with a coefficient of
**2.032968**.

By comparison, `attendance_pct` had a weaker correlation with exam score
(**r = 0.106649**) and a smaller regression coefficient
of **0.088653**. `sleep_hours_per_night` was essentially
uncorrelated with exam score (**r = -0.021898**) and its
regression coefficient is also close to zero.

Therefore, the main conclusion from Monday's correlation analysis is
supported by the Linear Regression model: **study hours are the strongest
continuous predictor of exam score in this dataset.**


## 6. Create the Classification Target

For the classification task, a student is labelled as having a distinction when
their `exam_score` is at least 85.

The new `distinction` target contains **1** for students meeting the threshold
and **0** otherwise.

In [22]:
# Create the binary classification target
students["distinction"] = (students["exam_score"] >= 85).astype(int)

students[["exam_score", "distinction"]].head()

,exam_score,distinction
0,100.0,1
1,72.2,0
2,67.2,0
3,100.0,1
4,100.0,1


## 6.1 Check the Class Balance

In [23]:
distinction_fraction = students["distinction"].mean()
distinction_count = students["distinction"].sum()
non_distinction_count = (students["distinction"] == 0).sum()

classification_summary = f"""
### 6.2 Finding

There are **{distinction_count:,} students** who meet the distinction
threshold and **{non_distinction_count:,} students** who do not.

Therefore, **{distinction_fraction:.6%}** of students have an exam score
of at least 85.

This shows how common the positive class is in the dataset. If the classes
are substantially imbalanced, a model can achieve high accuracy simply by
predicting the majority class most of the time. This is why accuracy must be
interpreted alongside precision, recall, and F1 score.
"""

display(Markdown(classification_summary))


### 6.2 Finding

There are **392 students** who meet the distinction
threshold and **208 students** who do not.

Therefore, **65.333333%** of students have an exam score
of at least 85.

This shows how common the positive class is in the dataset. If the classes
are substantially imbalanced, a model can achieve high accuracy simply by
predicting the majority class most of the time. This is why accuracy must be
interpreted alongside precision, recall, and F1 score.


## 7. Create a Fresh Stratified Train/Test Split

The classification target is different from the regression target, so a fresh
train/test split is created.

The split uses `stratify=y_classification` to preserve approximately the same
proportion of distinction and non-distinction students in both the training
and test sets.

In [24]:

X_classification = students[
    [
        "study_hours_per_week",
        "sleep_hours_per_night",
        "attendance_pct",
        "class_section"
    ]
].copy()

X_classification = pd.get_dummies(
    X_classification,
    columns=["class_section"],
    drop_first=True
)

y_classification = students["distinction"]

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_classification,
    y_classification,
    test_size=0.2,
    random_state=42,
    stratify=y_classification
)

## 7.1 Train the Classification Baseline

The classification baseline uses `DummyClassifier(strategy="most_frequent")`.
It always predicts the most common class observed in the training data.

This provides the minimum reference point that the Logistic Regression model
should improve upon.

In [25]:

classification_baseline = DummyClassifier(
    strategy="most_frequent"
)

classification_baseline.fit(
    X_train_class,
    y_train_class
)

baseline_class_predictions = classification_baseline.predict(
    X_test_class
)

In [26]:
baseline_accuracy = accuracy_score(
    y_test_class,
    baseline_class_predictions
)

baseline_precision = precision_score(
    y_test_class,
    baseline_class_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    y_test_class,
    baseline_class_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test_class,
    baseline_class_predictions,
    zero_division=0
)

In [27]:
classification_baseline_finding = f"""
### 7.2 Finding

The majority-class baseline achieved an **accuracy of {baseline_accuracy:.6f}**,
**precision of {baseline_precision:.6f}**, **recall of {baseline_recall:.6f}**,
and **F1 score of {baseline_f1:.6f}** on the test set.

**Accuracy alone would be misleading because the baseline predicts only the
majority class, so its accuracy mainly reflects how common that class is rather
than whether the model can successfully identify distinction students.**
"""

display(Markdown(classification_baseline_finding))


### 7.2 Finding

The majority-class baseline achieved an **accuracy of 0.650000**,
**precision of 0.650000**, **recall of 1.000000**,
and **F1 score of 0.787879** on the test set.

**Accuracy alone would be misleading because the baseline predicts only the
majority class, so its accuracy mainly reflects how common that class is rather
than whether the model can successfully identify distinction students.**


## 8. Train the Logistic Regression Model

Logistic Regression is used here as the real classification model. It learns
the relationship between the input features and the binary `distinction`
target.

The model is trained on the same training data used by the classification
baseline so that the comparison is fair.

In [28]:

logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logistic_model.fit(
    X_train_class,
    y_train_class
)

logistic_predictions = logistic_model.predict(
    X_test_class
)

In [29]:
logistic_accuracy = accuracy_score(
    y_test_class,
    logistic_predictions
)

logistic_precision = precision_score(
    y_test_class,
    logistic_predictions,
    zero_division=0
)

logistic_recall = recall_score(
    y_test_class,
    logistic_predictions,
    zero_division=0
)

logistic_f1 = f1_score(
    y_test_class,
    logistic_predictions,
    zero_division=0
)

In [30]:
accuracy_improvement = logistic_accuracy - baseline_accuracy
precision_improvement = logistic_precision - baseline_precision
recall_improvement = logistic_recall - baseline_recall
f1_improvement = logistic_f1 - baseline_f1

In [31]:
logistic_comparison = f"""
### 8.1 Finding

The Logistic Regression model achieved an **accuracy of {logistic_accuracy:.6f}**,
**precision of {logistic_precision:.6f}**, **recall of {logistic_recall:.6f}**,
and **F1 score of {logistic_f1:.6f}**.

Compared with the majority-class baseline, Logistic Regression changed
accuracy by **{accuracy_improvement:+.6f}**, precision by
**{precision_improvement:+.6f}**, recall by **{recall_improvement:+.6f}**,
and F1 by **{f1_improvement:+.6f}**.

The model improved accuracy, precision, and F1, while recall decreased by
**{abs(recall_improvement):.6f}**. This shows that Logistic Regression
provides a more balanced classification than simply predicting the majority
class, rather than improving every metric simultaneously.
"""

display(Markdown(logistic_comparison))


### 8.1 Finding

The Logistic Regression model achieved an **accuracy of 0.766667**,
**precision of 0.812500**, **recall of 0.833333**,
and **F1 score of 0.822785**.

Compared with the majority-class baseline, Logistic Regression changed
accuracy by **+0.116667**, precision by
**+0.162500**, recall by **-0.166667**,
and F1 by **+0.034906**.

The model improved accuracy, precision, and F1, while recall decreased by
**0.166667**. This shows that Logistic Regression
provides a more balanced classification than simply predicting the majority
class, rather than improving every metric simultaneously.


# 9. Reproducibility Check

The complete notebook is organized as a single reproducible ML pipeline.

The workflow starts with deterministic dataset generation, applies feature
engineering, creates fixed train/test splits, trains regression and
classification baselines, trains the real models, evaluates their performance,
and interprets the results.

All random operations use fixed seeds so that the same results are obtained
when the notebook is executed from a fresh kernel.

## 9.1 Check Random Seeds

The synthetic dataset uses `seed=21`, while the train/test splits use
`random_state=42`.

The regression split is created without stratification because `exam_score`
is continuous. The classification split uses `stratify=y_classification` so
that the distinction/non-distinction class proportions are preserved in both
the training and test sets.

The Logistic Regression model also uses a fixed `random_state=42`.

## 9.2 Final Pipeline Verification

The notebook should execute successfully from the first cell to the last
without requiring cells to be run manually in a different order.

The final verification checks that both real models have been trained and that
their evaluation metrics are available.

In [32]:
pipeline_verification = f"""
### 9.3 Finding

The regression pipeline produced a Linear Regression **RMSE of
{linear_rmse:.6f}** and **R² of {linear_r2:.6f}**.

The classification pipeline produced a Logistic Regression **accuracy of
{logistic_accuracy:.6f}**, **precision of {logistic_precision:.6f}**,
**recall of {logistic_recall:.6f}**, and **F1 score of {logistic_f1:.6f}**.

All required models, predictions, and evaluation metrics are available after
executing the notebook from top to bottom.
"""

display(Markdown(pipeline_verification))


### 9.3 Finding

The regression pipeline produced a Linear Regression **RMSE of
7.058469** and **R² of 0.525687**.

The classification pipeline produced a Logistic Regression **accuracy of
0.766667**, **precision of 0.812500**,
**recall of 0.833333**, and **F1 score of 0.822785**.

All required models, predictions, and evaluation metrics are available after
executing the notebook from top to bottom.
